In [1]:

import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Load and preprocess data
df = pd.read_csv("LP IV Updated Datasets/AutoEncoder/ECG/ecg.csv", header=None)

train, test = train_test_split(
    df, test_size=0.2, stratify=df[140]
)

scaler = MinMaxScaler()
x_train = scaler.fit_transform(train[train[140] == 1].iloc[0:, 0:140])
x_test = scaler.transform(test.iloc[0:, 0:140])

# Create Autoencoder
model = Sequential([
    Input(shape=(140,)),
    Dense(16, activation='relu'),
    Dense(140, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='mse'
)

# Train
model.fit(x_train, x_train, epochs=10, batch_size=128)

# Detect anomalies
error = np.mean((x_train - model.predict(x_train)) ** 2, axis=1)
threshold = np.percentile(error, 95)

error = np.mean((x_test - model.predict(x_test)) ** 2, axis=1)
anomalies = error > threshold

print("Anomalies:", anomalies.sum())


Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0239
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0202 
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0152 
Epoch 4/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0115 
Epoch 5/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0095 
Epoch 6/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0085 
Epoch 7/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0079 
Epoch 8/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0074 
Epoch 9/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0070 
Epoch 10/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0067
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Anomalies: 423
